In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforw

In [4]:
import os

from transformers_sae.ops import load_saes

# replace with appropriate directory for your system, if needed
CHECKPOINT_BASE_PATH = (
    f"{os.getenv('BLOG_POST_BUCKET_LOCAL')}/gemma_2_2b/sae_checkpoints"
)

saes = load_saes(
    f"{CHECKPOINT_BASE_PATH}/next_layer_in_place_finetuned_lista_unit_scale",
    range(model.num_layers),
)
for sae in saes.values():
    sae.eval()
    sae.onload()


Loading thresholds from /Volumes/MacData/blog_post_bucket/gemma_2_2b/sae_checkpoints/next_layer_in_place_finetuned_lista_unit_scale/train_thresholds_0
Loaded checkpoint for layer 5
Updated thresholds for layer 5
Loaded checkpoint for layer 9
Updated thresholds for layer 9
Loaded checkpoint for layer 11
Updated thresholds for layer 11
Loaded checkpoint for layer 8
Updated thresholds for layer 8
Loaded checkpoint for layer 3
Updated thresholds for layer 3
Loaded checkpoint for layer 10
Updated thresholds for layer 10
Loaded checkpoint for layer 6
Updated thresholds for layer 6
Loaded checkpoint for layer 7
Updated thresholds for layer 7
Loaded checkpoint for layer 4
Updated thresholds for layer 4
Loaded checkpoint for layer 0
Updated thresholds for layer 0
Loaded checkpoint for layer 1
Updated thresholds for layer 1
Loaded checkpoint for layer 2
Updated thresholds for layer 2
Loaded checkpoint for layer 17
Updated thresholds for layer 17
Loaded checkpoint for layer 13
Updated thresholds 

In [ ]:
# This will run (though slowly) on an M1 Pro. CUDA is highly recommended for anything more complicated.

from transformers_sae.ops import generate
from transformers_sae.replacement_model import make_replacement_model

replacement_model = make_replacement_model(model, saes)

with torch.autocast(
    device_type="cuda" if model.device.type == "cuda" else "cpu",
    dtype=torch.bfloat16,
):
    generate(
        "The capital of France,",
        replacement_model,
        tokenizer,
        max_new_tokens=100,
    )

 Paris, is a beautiful city with a lot of history and a lot of history. The city is a great place to go for a while, and it is a great place to go for a while. The city is a great place to go for a time, and it is a great place to go for a time. The city is a great place to go for a time, and it is a great place to go for a time. The city is a great place to go for a time


In [ ]:
from transformers_sae.tokenization import make_dataloader
from transformers_sae.activation_data import make_activation_batch

# Example: get feature activations for the 12th layer from the first context_length tokens in the validation set.

TARGET_LAYER = 12
for batch in make_dataloader(
    model,
    tokenizer,
    validation_dataset,
    max_tokens=model.context_length,
    tokenizer_batch_size=256,
    inference_batch_size=1,
):
    print("Actual num tokens: ", batch.num_tokens)
    batch.to(model.device)
    with (
        torch.no_grad(),
        torch.autocast(
            device_type="cuda" if model.device.type == "cuda" else "cpu",
            dtype=torch.bfloat16,
        ),
    ):
        activations = make_activation_batch(
            replacement_model,
            [(TARGET_LAYER, "sae")],
            batch,
            end_layer=TARGET_LAYER + 1,
        )
        # NB: the sae_features tensor is "squeezed" such that it does not include values for special tokens
        # (such as the bos token and padding); this may change in a later release, as it can be a bit
        # annoying to correlate with specific token positions.
        active_features = activations[TARGET_LAYER].sae_features.nonzero(as_tuple=True)
        print(active_features)
        print(activations[TARGET_LAYER].sae_features[active_features])


Actual num tokens:  1024
(tensor([0, 0, 0,  ..., 0, 0, 0], device='mps:0'), tensor([   0,    0,    0,  ..., 1022, 1022, 1022], device='mps:0'), tensor([  227,   620,   704,  ..., 16181, 16235, 16354], device='mps:0'))
tensor([2.9219, 7.4062, 1.0625,  ..., 2.2344, 5.6562, 1.0078], device='mps:0',
       dtype=torch.bfloat16)
